## Load the summary information and preprocessed data

In [1]:
import os
import json
import pandas as pd

from pathlib import Path

from single_sample_gsea import ss_gsea


# Get GEO_ID from folder name (same pattern as prepare_data.ipynb)
GEO_ID = os.path.basename(os.path.dirname(os.path.abspath('__file__')))


description_path = Path('description.json')
preprocessed_csv_path = Path(f"{GEO_ID}_preprocessed.csv")

if not description_path.exists():
    raise FileNotFoundError(
        "description.json not found. Please run 'prepare_data.ipynb' first to generate it."
    )

if not preprocessed_csv_path.exists():
    raise FileNotFoundError(
        f"{preprocessed_csv_path} not found. Please run 'preprocess_data.ipynb' first to generate it."
    )

with open(description_path) as f:
    summary_data = json.load(f)

df = pd.read_csv(preprocessed_csv_path)

print("Data contains", df.shape[0], "samples and", df.shape[1], "columns.")
print("First few rows of the data:")
df.head()


Data contains 58 samples and 12013 columns.
First few rows of the data:


,overall survival (months),vital status,sample_title,tissue,unique sample id,patient id,grade,batch,idh status prediction,age,...,COX2,ATP8,ATP6,COX3,ND3,ND4L,ND4,ND5,ND6,CYTB
0,115.733333,1,Sample_S-10-33895_S22_L003merged,Glioma,Sample_S-10-33895,1,HGG,1,Mutant,22.0,...,6.416128,4.085363,5.955792,6.335819,4.288661,5.358118,6.942101,6.587702,5.318951,6.187649
1,26.833333,1,Sample_R378-14-133900_S12_L002,Glioma,Sample_14-133900,10,HGG,2,WT,39.0,...,6.874357,3.695918,5.518125,6.827268,4.515771,5.706525,7.234463,6.714490,4.857237,6.149874
2,34.433333,1,Sample_S-12-53702_S22_L003merged,Glioma,Sample_S-12-53702,103,HGG,1,WT,59.0,...,6.856569,4.238486,6.308461,7.027931,4.694066,5.504444,7.328217,7.096958,5.321711,6.411636
3,182.466667,1,Sample_S-13-16523_S22_L003merged,Glioma,Sample_S-13-16523,104,HGG,1,Mutant,31.0,...,7.261454,4.524313,6.533104,7.372353,4.526102,5.855307,7.544500,7.244535,5.657381,6.958492
4,8.866667,0,Sample_R378-15-49530_S13_L002,Glioma,Sample_15-49530,11,HGG,2,Mutant,49.0,...,5.670679,3.080187,4.963198,5.792980,3.826287,5.216755,6.434277,6.151793,4.775085,5.818796


## Load gene sets

In [2]:
MSIGDB_JSON = Path("../../resources/h.all.v2026.1.Hs.json")

with MSIGDB_JSON.open(encoding="utf-8") as f:
    gene_sets = {name: data["geneSymbols"] for name, data in json.load(f).items()}

print(f"Loaded {len(gene_sets)} gene sets.")


Loaded 50 gene sets.


## Compute single sample GSEA on the preprocessed data

Select only gene columns for the computation.

In [3]:
# IMPORTANT: This is dataset-specific and is the only variable that needs to be changed for each of the dataset.
sample_id_column = 'sample_title'

gene_expression_df = df.iloc[:, summary_data['data_summary']['clinical_features']:].copy()
gene_expression_df.index = df[sample_id_column].astype(str)
gene_expression_df = gene_expression_df.T
gene_expression_df.index.name = "gene"

result = ss_gsea(gene_expression_df, gene_sets)


## Merge the results with the clinical data

The output table should have the same number of rows as before and columns should sum up to clinical data + 50 gene sets.

In [4]:
clinical_df = df.iloc[:, :summary_data["data_summary"]["clinical_features"]].copy()
clinical_df = clinical_df.set_index(sample_id_column)

output_df = pd.concat([clinical_df, result], axis=1)
output_df = output_df.reset_index(drop=False)

output_df.to_csv(f"{GEO_ID}_preprocessed_ssgsea.csv", index=False)

# sanity check (there is 50 gene sets)
assert output_df.shape[1] == summary_data["data_summary"]["clinical_features"] + 50
